In [1]:
from collections import Counter, defaultdict
import time
import os
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools
import random as r
import statistics as stats
from docx import Document
from tabulate import tabulate
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (auc, average_precision_score, 
                              roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression
import seaborn as sns
from sklearn.base import BaseEstimator

import warnings
warnings.filterwarnings('ignore')


plt.rcParams.update({'font.size': 12})

# Análisis de las variables: aproximación con Grafos

### Objetivo
En este archivo, el objetivo es armar redes para cada una de las variables, tomando 10 usuarios aleatorios entre los que tienen algún nodo fraude. 

Fase de análisis y exploración de variables, ya que tengo 64 variables y quiero elegir entre ellas las que aporten mayor información a los algoritmos. 

A través de la observación de redes y el grado de los nodos, es que se pretende obtener las variables que mejor distingan los nodos fraude de los que no. 

### Output del archivo
Corriendo todos los kernel del archivo, se genera un folder llamado 'Output_variables', en donde se guardan:

- Archivo "tablas_métricas.txt" con las métricas de las variables correspondientes a cada red generada:
   Grado promedio de la red, grado promedio de nodos fraude, grado promedio nodos no fraude. 
- 1280 grafos: todos los grafos correspondientes a las redes

#### 1. Cargo la data
    1.1. Guardo todo como str 

    1.2. Relleno los null con '9999' ya que me interesa que relacione null
    
    1.3. Creo dos variables nuevas a partir de la concatenacion de variables disponibles
    
    1.3. Tomo las variables para las que quiero hacer el análisis


In [2]:
# genero el DataFrame "data" con la data principal, que son los usuarios que presentan casos de fraude

data = pd.read_excel(open('raw_data_estafas.xlsx', 'rb'),
                     converters = {
                         'hora_evento':str, 'user_tuvo_estafa':str, 'Label':str, 'id_evento':str, 'id_usuario':str, 'id_sesion':str,
                         'fecha':str, 'hash_usuario':str, 'tipo_evento':str, 'score':str, 'regla_1_riesgo':str,
                         'score_1_riesgo':str, 'regla_2_riesgo':str, 'score_2_riesgo':str, 'regla_3_riesgo':str,
                         'score_3_riesgo':str, 'regla_4_riesgo':str, 'score_4_riesgo':str, 'decision':str, 'cookie':str,
                         'agente_busqueda_hash':str, 'huella_software_hash':str, 'plugin_navegador_hash':str,
                         'pantalla_hash':str, 'lenguaje_sistema_operativo':str, 'lenguaje_usuario':str, 'lenguaje_navegador_detalles':str,
                         'lenguaje_navegador':str, 'zona_horaria':str, 'ip_address':str, 'ip_pais':str, 'ip_region':str,
                         'ip_ciudad':str, 'ip_isp':str, 'canal':str, 'cookie_persistent':str, 'data_1':str,
                         'data_2':str, 'data_4':str, 'data_5':str, 'data_7':str, 'data_8':str, 'data_9':str, 
                         'data_13':str, 'data_29':str, 'data_30':str, 'data_31':str, 'data_37':str, 'data_40':str,
                         'data_41':str, 'data_42':str, 'data_43':str, 'data_44':str, 'data_45':str, 'data_56':str,
                         'data_60':str, 'data_77':str, 'data_78':str, 'data_80':str, 'data_82':str, 'data_83':str,
                         'data_3':str, 'antiguedad_cookies':str, 'data_100':str, 'data_107':str, 'antiguedad_device':str, 'data_115':str, 
                         'velocity_isp_web_10d':str, 'velocity_isp_web_30d':str, 'velocity_isp_mobile_10d':str,
                         'velocity_isp_mobile_30d':str, 'antiguedad_geodata':str, 'sistema_operativo':str, 'navegador_nombre':str,
                         'navegador_version':str, 'rdp_trojan_collection_status':str, 'ip_isp_nombre':str
                     }
                    )


# relleno los null con un valor, para que no de error, ademas me interesa que los null compartan eje
data.fillna('9999', inplace = True) 


# genero la variable 'navegador_caract que concatena  'navegador_nombre' con 'navegador_version'
data['navegador_caract'] = data['navegador_nombre'] + data['navegador_version']

# tambien genero una nueva variable user_caract a partir de las 4 variables HASH: 
data['user_caract'] = data['agente_busqueda_hash'] + '_'+ data['huella_software_hash'] + '_' + data['plugin_navegador_hash'] + '_'+ data['pantalla_hash']

In [3]:
data.head()

,hora_evento,user_tuvo_estafa,Label,id_evento,id_usuario,id_sesion,fecha,hash_usuario,tipo_evento,score,...,velocity_isp_mobile_10d,velocity_isp_mobile_30d,antiguedad_geodata,sistema_operativo,navegador_nombre,navegador_version,rdp_trojan_collection_status,ip_isp_nombre,navegador_caract,user_caract
0,2021-01-29 23:40:21.070,1,0,6b62:b1282bf4771:b4efd5f_TRX,150073,5b62:b1282bf4771:b4efd5f,20210129,717,SESSION_SIGNIN,29,...,0,0,9999,Windows,Chrome,87,1,Telecom Argentina S.A.,Chrome87,5b3c11bff4_6b9f81ec42_337e07ddb4_b9236421dc
1,2021-01-29 19:16:28.447,1,0,2f32:6c9bcee4771:2b8d87c1-_TRX,150073,1f32:6c9bcee4771:2b8d87c1-,20210129,717,SESSION_SIGNIN,29,...,0,0,9999,Windows,Chrome,87,1,Telecom Argentina S.A.,Chrome87,5b3c11bff4_6b9f81ec42_337e07ddb4_b9236421dc
2,2021-01-29 18:05:08.533,1,0,c753:c9bd3ce4771:de07ff07_TRX,150073,b753:c9bd3ce4771:de07ff07,20210129,717,SESSION_SIGNIN,29,...,0,0,9999,Windows,Chrome,87,1,Telecom Argentina S.A.,Chrome87,5b3c11bff4_6b9f81ec42_337e07ddb4_b9236421dc
3,2021-01-28 17:55:59.533,1,0,5715-:31025f94771:b4efd5f_TRX,150073,6715-:31025f94771:b4efd5f,20210128,717,SESSION_SIGNIN,29,...,0,0,9999,Windows,Chrome,87,1,Telecom Argentina S.A.,Chrome87,5b3c11bff4_6b9f81ec42_337e07ddb4_b9236421dc
4,2021-01-27 22:33:17.617,1,0,a586:ce533e44771:b4efd5f_TRX,150073,9586:ce533e44771:b4efd5f,20210127,717,SESSION_SIGNIN,30,...,0,0,9999,Windows,Chrome,87,1,Telecom Argentina S.A.,Chrome87,5b3c11bff4_6b9f81ec42_337e07ddb4_b9236421dc


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5685 entries, 0 to 5684
Data columns (total 79 columns):
hora_evento                     5685 non-null object
user_tuvo_estafa                5685 non-null object
Label                           5685 non-null object
id_evento                       5685 non-null object
id_usuario                      5685 non-null object
id_sesion                       5685 non-null object
fecha                           5685 non-null object
hash_usuario                    5685 non-null object
tipo_evento                     5685 non-null object
score                           5685 non-null object
regla_1_riesgo                  5685 non-null object
score_1_riesgo                  5685 non-null object
regla_2_riesgo                  5685 non-null object
score_2_riesgo                  5685 non-null object
regla_3_riesgo                  5685 non-null object
score_3_riesgo                  5685 non-null object
regla_4_riesgo                  5685 non-null

In [5]:
vars_for_grapho = ['hash_usuario', 'regla_1_riesgo', 'regla_2_riesgo', 'regla_3_riesgo', 'regla_4_riesgo', 'cookie',
        'agente_busqueda_hash', 'huella_software_hash', 'plugin_navegador_hash', 'pantalla_hash', 'lenguaje_sistema_operativo',
       'lenguaje_usuario', 'lenguaje_navegador_detalles', 'lenguaje_navegador', 'zona_horaria',
       'ip_pais', 'ip_region', 'ip_ciudad', 'ip_isp',
       'canal', 'cookie_persistent', 'data_1', 'data_2',
       'data_4', 'data_5', 'data_7', 'data_8', 'data_9', 'data_13',
       'data_29', 'data_30', 'data_31', 'data_37', 'data_40',
       'data_41', 'data_42', 'data_43', 'data_44', 'data_45',
       'data_56', 'data_60', 'data_77', 'data_78', 'data_80',
       'data_82', 'data_83', 'data_3', 'antiguedad_cookies', 'data_100',
       'data_107', 'antiguedad_device', 'data_115', 'velocity_isp_web_10d',
       'velocity_isp_web_30d', 'velocity_isp_mobile_10d',
       'velocity_isp_mobile_30d', 'antiguedad_geodata', 'sistema_operativo',
       'navegador_nombre', 'navegador_version', 'rdp_trojan_collection_status',
       'navegador_caract','user_caract', "ip_address"]
       

#### 2. Funciones armado y guardado de Redes

    2.1. Función data_for_grapho(): Transforma la estructura de la data, para prepararla para armar una red
    
    2.2. Función carga_nodos(): Función auxiliar que toma un dataset preparado y carga los nodos correspondientes a una red
    
    2.3. Función carga_ejes(): Función auxiliar que toma un dataset preparado y carga los ejes correspondientes a una red
    
    2.4. Función metricas_grapho(): Toma una red y devuelve una tabla con sus métricas (Grado promedio red/fraude/no fraude)
    
    2.5. Función carga_grapho(): Utiliza las funciones anteriores para el armado de un grafo y su guardado
    
    2.6. Función graphos_users(): Corre la función 2.5 para una lista de usuarios definida
    
    2.7. Función regla_decision_variables(): Toma las métricas de la variable y define si es útil o no
    
    2.8. Función graphos_vars(): Arma los grafos para una lista de usuarios y de variables definidos    


In [6]:
def data_for_grapho(variable,user_id,raw_data):
    # Esta funcion toma el dataset crudo y devuelve los datos preparados para armar un grapho, 
    # para un user_id y una variable determinados
    
    # El input es el data_set recien cargado y devuelve un nuevo df: nodos_dataset que sirve para armar el grapho:
    # nodos_dataset['index'] es el id del nodo, lo que voy a cargar como nodos en el Grapho
    # nodos_dataset['Label'] es el atributo del nodo (si corresponde a un fraude o no)
    
    user_id   = str(user_id)
    user_dataset = raw_data[raw_data['id_usuario'] == user_id]
    print("Data for user '{}',  con {} observaciones, {} fraudes".format(user_id,len(user_dataset),
                                                                         len(user_dataset[user_dataset['Label'] == '1'])))
    
    variable = str(variable)
    variables = [variable, 'Label']
    nodos_dataset = user_dataset[variables] 
    nodos_dataset = nodos_dataset.reset_index()
    nodos_dataset.rename({'Label': 'Fraude','index': 'Label'}, axis=1, inplace=True)
    
    return nodos_dataset           

In [7]:
def carga_nodos(nodos_dataset, red):
    # Toma el dataset armado para el grapho y carga los nodos correspondientes, con su atributo

    for i in nodos_dataset['Label']:
        # cargo el nodo 
        red.add_node(i)
        # le agrego el atributo
        red.nodes[i]['Fraude'] = int(nodos_dataset[nodos_dataset['Label'] == i]['Fraude'])

    print("Se agregaron {} nodos a la red".format(len(red.nodes)))

In [8]:
def carga_ejes(nodos_dataset,variable,red):
    # Toma el dataset armado para el grapho y carga los ejes correspondientes
    
    #    saco los valores únicos para la variable en cuestión
    values = list(np.unique(nodos_dataset[variable].values))  
    #    carga de ejes
    for value in values:    
        # genero la lista de nodos que comparten un mismo valor para la variable en cuestion
        list_nodos = list(nodos_dataset[nodos_dataset[variable] == value]['Label'].values)
        
        # genero todas las combinaciones de valores en un la lista list_nodos, para luego cargar los ejes 
        if len(list_nodos) > 1:
            combinations = set(itertools.combinations(list_nodos, 2))
            list_edges = list(combinations)

            # agrego el eje al grapho: cada tupla representa que esos dos nodos comparten valor, entonces sera un eje
            red.add_edges_from(list_edges)

    print("Se agregaron {} ejes a la red".format(len(red.edges)))
    

In [9]:
def metricas_grapho(grapho):
    # Toma el grafo y calcula las métricas deseadas
    # Funcion que calcula las metricas de una red: 
    #        - Grado promedio del grafo total
    #        - Grado promedio nodos fraude
    #        - Primer decil de grado nodos no fraude
    # Devuelve una tupla de len 3, con estos tres valores

    
    # grado promedio generico toda la red
    mean_degree = str(round(stats.mean(dict(grapho.degree()).values()))).rjust(14)

    # grado promedio nodos fraude
    mean_degree_f = str(round(stats.mean([grapho.degree(node[0]) for node in grapho.nodes(data=True) if node[1]['Fraude']==1]))).rjust(14)
    
    # Primer decil del gr nodos no fraude
    first_decil_nf = str(round(np.percentile([grapho.degree(node[0]) for node in grapho.nodes(data=True) if node[1]['Fraude']==0],10))).rjust(14)

    return (mean_degree,mean_degree_f,first_decil_nf)


In [10]:
def carga_grapho(variable,user_id,raw_dataset):
    # Esta función toma el dataset original, llama a las funciones 1 y 2 para la preparacion de datos. 
    # Guarda el grapho para la variable y user correspondiente.
    # Devuelve una tupla de metricas para el grapho, que serán utilizados para estudiar la variable más adelante
    # Output: tupla de len 3, con los tres valores de las métricas del grafo cargado
    # 1. creo el grapho 
    red = nx.Graph()

    # 2. preparo la data
    nodos_dataset = data_for_grapho(variable,user_id,raw_dataset)

    # 3. carga de nodos con su respectivo atributo 'Label'
    carga_nodos(nodos_dataset,red)

    # 4. carga los ejes
    carga_ejes(nodos_dataset,variable,red)

    # 5. guarda el grapho con su variable y user en el nombre, en una carpeta llamada Graphos_variables
    name = "red_{}_{}".format(variable,user_id)
    if not os.path.exists('Output_variables'):
        os.makedirs('Output_variables')
    Path = os.getcwd()+'\\Output_variables\\'
    nx.write_gml(red, str(Path)+str(name)+'.gml')
    
    # 6. métricas del grapho: Grado promedio red, grado promedio nodos fraude, primer decil grados nodos no fraude
    #    Es una lista de len 3 con las tres métricas
    metrics = metricas_grapho(red)

    print("Se guardo el Grapho {}.gml'".format(name))
    return metrics

In [11]:
def graphos_users(variable,users_list,dataset):
    # Para una variable determinada, arma grafos para todos los users de una lista de users
    # Output: un dict donde la clave es el usuario y el valor es la lista de metricas del grafo correspondiente, para la variable
    
    metricas_users = dict()
    for user in users_list:
        print("User {}".format(user))
        metrics = carga_grapho(variable,user,dataset)
        key = str(user).rjust(14)
        metricas_users[key] = metrics
    print(metricas_users)
    
    return metricas_users

In [12]:
def regla_decision_variables(variable,tabla_metricas):
    # Toma las métricas para la variable y define si es útil o no, según criterio definido
    # Output: lista len 4, con info de la variable, los promedios de las métricas, y la definicion de si es útil o no 
    
    sum_GP_fraude  = 0
    sum_1D_Nfraude = 0
    conteo = 0
    variable_util = ''
    
    for user in tabla_metricas:
        tupla_values = tabla_metricas[user]
        tupla_values_int = tuple(int(float(x)) for x in tupla_values)
        conteo += 1 
        sum_GP_fraude  += tupla_values_int[1]
        sum_1D_Nfraude += tupla_values_int[2]   

    # toma los grados promedio de los nodos fraude, para todos los usuarios, y hace un promedio    
    promedio_GP_fraude  = (sum_GP_fraude/conteo)
    
    # toma los priemros deciles de grados de los nodos genuinos, para todos los usuarios, y hace un promedio  
    promedio_1DG_Nfraude = (sum_1D_Nfraude/conteo)
    
    # calcula la variacion entre estas dos métricas:
    if promedio_1DG_Nfraude != 0:  
        variac_G = ((promedio_GP_fraude/promedio_1DG_Nfraude)-1)*(-1)
    else:
        variac_G = 0

    if variac_G >= 0.8:
        variable_util = 'Y'
    elif variac_G <= 0.5:
        variable_util = 'N'
    else:
        variable_util = 'D'
    
    info_final = [variable,promedio_GP_fraude,promedio_1DG_Nfraude,variable_util]
    
    return info_final

In [13]:
def graphos_vars(vars_for_grapho,users_list,dataset):
    # Para una lista de variables y una lista de users, arma los graphos y corre las funciones anteriores. Salva archivos de grafos
    # y archivos csv con la info por variable.
    
    if not os.path.exists('Output_variables'):
        os.makedirs('Output_variables')
    file1 = open(os.getcwd() + "\\Output_variables\\tablas_metricas.txt","w") 
    file2 = open(os.getcwd() + "\\Output_variables\\variables_utiles.csv","w",newline='')
    
    file_2_writer = csv.writer(file2, delimiter=',')
    headers = ['variable','promedio_GP_fraude','promedio_1DG_Nfraude','variable_util']
    file_2_writer.writerow(headers)
    
    
    for variable in vars_for_grapho:
        print("VARIABLE {}".format(variable))
        tabla_metricas = graphos_users(variable,users_list,dataset)
        
        # Armo la tabla que quiero printear despues, para el analisis
        msg ='\n\n\n' + "METRICAS VARIABLE: '{}' ".format(variable) + '\n\n'
        print(msg)
        headers = ['User', 'GP Red', 'GP Fraude', 'Prim. decil NF']
        tabla = tabulate([(k,) + v  for k,v in tabla_metricas.items()], headers = headers, 
                         numalign="center", stralign="center",tablefmt="psql")
        print(tabla)
        
        # Voy guardando las tablas en un txt
        file1.write(msg)
        file1.write(tabla)
        
        # Variables utiles
        var_utiles = regla_decision_variables(variable,tabla_metricas)
        file_2_writer.writerow(var_utiles)
        
    
    file1.close()
    file2.close()

#### 4. Armado de grafos/redes

Finalmente, selecciono random 20 users de la lista de usuarios, y armo los grafos para todas las variables para los 10 users. 

In [14]:
all_users = list(data['id_usuario'].unique())
#r.seed(10)
users_list = r.sample(all_users,20)
users_list

['4699985',
 '2400976',
 '4285617',
 '4301928',
 '7669090',
 '150073',
 '3408431',
 '4289668',
 '6726081',
 '4414533',
 '3725780',
 '5416930',
 '3068918',
 '7575855',
 '4502203',
 '6619023',
 '3782790',
 '2778561',
 '3617729',
 '3935173']

In [15]:
graphos_vars(vars_for_grapho,users_list,data)

VARIABLE hash_usuario
User 4699985
Data for user '4699985',  con 164 observaciones, 2 fraudes
Se agregaron 164 nodos a la red
Se agregaron 13366 ejes a la red
Se guardo el Grapho red_hash_usuario_4699985.gml'
User 2400976
Data for user '2400976',  con 60 observaciones, 2 fraudes
Se agregaron 60 nodos a la red
Se agregaron 1770 ejes a la red
Se guardo el Grapho red_hash_usuario_2400976.gml'
User 4285617
Data for user '4285617',  con 126 observaciones, 1 fraudes
Se agregaron 126 nodos a la red
Se agregaron 7875 ejes a la red
Se guardo el Grapho red_hash_usuario_4285617.gml'
User 4301928
Data for user '4301928',  con 175 observaciones, 2 fraudes
Se agregaron 175 nodos a la red
Se agregaron 15225 ejes a la red
Se guardo el Grapho red_hash_usuario_4301928.gml'
User 7669090
Data for user '7669090',  con 15 observaciones, 1 fraudes
Se agregaron 15 nodos a la red
Se agregaron 105 ejes a la red
Se guardo el Grapho red_hash_usuario_7669090.gml'
User 150073
Data for user '150073',  con 195 observ

Se agregaron 105 nodos a la red
Se agregaron 2286 ejes a la red
Se guardo el Grapho red_regla_1_riesgo_5416930.gml'
User 3068918
Data for user '3068918',  con 161 observaciones, 3 fraudes
Se agregaron 161 nodos a la red
Se agregaron 2883 ejes a la red
Se guardo el Grapho red_regla_1_riesgo_3068918.gml'
User 7575855
Data for user '7575855',  con 24 observaciones, 1 fraudes
Se agregaron 24 nodos a la red
Se agregaron 35 ejes a la red
Se guardo el Grapho red_regla_1_riesgo_7575855.gml'
User 4502203
Data for user '4502203',  con 28 observaciones, 2 fraudes
Se agregaron 28 nodos a la red
Se agregaron 92 ejes a la red
Se guardo el Grapho red_regla_1_riesgo_4502203.gml'
User 6619023
Data for user '6619023',  con 47 observaciones, 2 fraudes
Se agregaron 47 nodos a la red
Se agregaron 490 ejes a la red
Se guardo el Grapho red_regla_1_riesgo_6619023.gml'
User 3782790
Data for user '3782790',  con 176 observaciones, 3 fraudes
Se agregaron 176 nodos a la red
Se agregaron 2534 ejes a la red
Se guar

User 4285617
Data for user '4285617',  con 126 observaciones, 1 fraudes
Se agregaron 126 nodos a la red
Se agregaron 3042 ejes a la red
Se guardo el Grapho red_regla_3_riesgo_4285617.gml'
User 4301928
Data for user '4301928',  con 175 observaciones, 2 fraudes
Se agregaron 175 nodos a la red
Se agregaron 4067 ejes a la red
Se guardo el Grapho red_regla_3_riesgo_4301928.gml'
User 7669090
Data for user '7669090',  con 15 observaciones, 1 fraudes
Se agregaron 15 nodos a la red
Se agregaron 48 ejes a la red
Se guardo el Grapho red_regla_3_riesgo_7669090.gml'
User 150073
Data for user '150073',  con 195 observaciones, 1 fraudes
Se agregaron 195 nodos a la red
Se agregaron 8125 ejes a la red
Se guardo el Grapho red_regla_3_riesgo_150073.gml'
User 3408431
Data for user '3408431',  con 39 observaciones, 3 fraudes
Se agregaron 39 nodos a la red
Se agregaron 134 ejes a la red
Se guardo el Grapho red_regla_3_riesgo_3408431.gml'
User 4289668
Data for user '4289668',  con 362 observaciones, 2 fraude

Se guardo el Grapho red_regla_4_riesgo_3068918.gml'
User 7575855
Data for user '7575855',  con 24 observaciones, 1 fraudes
Se agregaron 24 nodos a la red
Se agregaron 30 ejes a la red
Se guardo el Grapho red_regla_4_riesgo_7575855.gml'
User 4502203
Data for user '4502203',  con 28 observaciones, 2 fraudes
Se agregaron 28 nodos a la red
Se agregaron 49 ejes a la red
Se guardo el Grapho red_regla_4_riesgo_4502203.gml'
User 6619023
Data for user '6619023',  con 47 observaciones, 2 fraudes
Se agregaron 47 nodos a la red
Se agregaron 219 ejes a la red
Se guardo el Grapho red_regla_4_riesgo_6619023.gml'
User 3782790
Data for user '3782790',  con 176 observaciones, 3 fraudes
Se agregaron 176 nodos a la red
Se agregaron 2029 ejes a la red
Se guardo el Grapho red_regla_4_riesgo_3782790.gml'
User 2778561
Data for user '2778561',  con 85 observaciones, 2 fraudes
Se agregaron 85 nodos a la red
Se agregaron 1176 ejes a la red
Se guardo el Grapho red_regla_4_riesgo_2778561.gml'
User 3617729
Data for

Se agregaron 126 nodos a la red
Se agregaron 1021 ejes a la red
Se guardo el Grapho red_agente_busqueda_hash_4285617.gml'
User 4301928
Data for user '4301928',  con 175 observaciones, 2 fraudes
Se agregaron 175 nodos a la red
Se agregaron 1383 ejes a la red
Se guardo el Grapho red_agente_busqueda_hash_4301928.gml'
User 7669090
Data for user '7669090',  con 15 observaciones, 1 fraudes
Se agregaron 15 nodos a la red
Se agregaron 25 ejes a la red
Se guardo el Grapho red_agente_busqueda_hash_7669090.gml'
User 150073
Data for user '150073',  con 195 observaciones, 1 fraudes
Se agregaron 195 nodos a la red
Se agregaron 3871 ejes a la red
Se guardo el Grapho red_agente_busqueda_hash_150073.gml'
User 3408431
Data for user '3408431',  con 39 observaciones, 3 fraudes
Se agregaron 39 nodos a la red
Se agregaron 89 ejes a la red
Se guardo el Grapho red_agente_busqueda_hash_3408431.gml'
User 4289668
Data for user '4289668',  con 362 observaciones, 2 fraudes
Se agregaron 362 nodos a la red
Se agrega

...

Se agregaron 2886 ejes a la red
Se guardo el Grapho red_sistema_operativo_5416930.gml'
User 3068918
Data for user '3068918',  con 161 observaciones, 3 fraudes
Se agregaron 161 nodos a la red
Se agregaron 7806 ejes a la red
Se guardo el Grapho red_sistema_operativo_3068918.gml'
User 7575855
Data for user '7575855',  con 24 observaciones, 1 fraudes
Se agregaron 24 nodos a la red
Se agregaron 141 ejes a la red
Se guardo el Grapho red_sistema_operativo_7575855.gml'
User 4502203
Data for user '4502203',  con 28 observaciones, 2 fraudes
Se agregaron 28 nodos a la red
Se agregaron 246 ejes a la red
Se guardo el Grapho red_sistema_operativo_4502203.gml'
User 6619023
Data for user '6619023',  con 47 observaciones, 2 fraudes
Se agregaron 47 nodos a la red
Se agregaron 531 ejes a la red
Se guardo el Grapho red_sistema_operativo_6619023.gml'
User 3782790
Data for user '3782790',  con 176 observaciones, 3 fraudes
Se agregaron 176 nodos a la red
Se agregaron 7949 ejes a la red
Se guardo el Grapho re

Se guardo el Grapho red_navegador_version_4699985.gml'
User 2400976
Data for user '2400976',  con 60 observaciones, 2 fraudes
Se agregaron 60 nodos a la red
Se agregaron 307 ejes a la red
Se guardo el Grapho red_navegador_version_2400976.gml'
User 4285617
Data for user '4285617',  con 126 observaciones, 1 fraudes
Se agregaron 126 nodos a la red
Se agregaron 2307 ejes a la red
Se guardo el Grapho red_navegador_version_4285617.gml'
User 4301928
Data for user '4301928',  con 175 observaciones, 2 fraudes
Se agregaron 175 nodos a la red
Se agregaron 4384 ejes a la red
Se guardo el Grapho red_navegador_version_4301928.gml'
User 7669090
Data for user '7669090',  con 15 observaciones, 1 fraudes
Se agregaron 15 nodos a la red
Se agregaron 42 ejes a la red
Se guardo el Grapho red_navegador_version_7669090.gml'
User 150073
Data for user '150073',  con 195 observaciones, 1 fraudes
Se agregaron 195 nodos a la red
Se agregaron 5465 ejes a la red
Se guardo el Grapho red_navegador_version_150073.gml'


Se agregaron 158 nodos a la red
Se agregaron 12403 ejes a la red
Se guardo el Grapho red_rdp_trojan_collection_status_3725780.gml'
User 5416930
Data for user '5416930',  con 105 observaciones, 1 fraudes
Se agregaron 105 nodos a la red
Se agregaron 3406 ejes a la red
Se guardo el Grapho red_rdp_trojan_collection_status_5416930.gml'
User 3068918
Data for user '3068918',  con 161 observaciones, 3 fraudes
Se agregaron 161 nodos a la red
Se agregaron 7522 ejes a la red
Se guardo el Grapho red_rdp_trojan_collection_status_3068918.gml'
User 7575855
Data for user '7575855',  con 24 observaciones, 1 fraudes
Se agregaron 24 nodos a la red
Se agregaron 157 ejes a la red
Se guardo el Grapho red_rdp_trojan_collection_status_7575855.gml'
User 4502203
Data for user '4502203',  con 28 observaciones, 2 fraudes
Se agregaron 28 nodos a la red
Se agregaron 218 ejes a la red
Se guardo el Grapho red_rdp_trojan_collection_status_4502203.gml'
User 6619023
Data for user '6619023',  con 47 observaciones, 2 frau

Se agregaron 146 nodos a la red
Se agregaron 1448 ejes a la red
Se guardo el Grapho red_navegador_caract_3617729.gml'
User 3935173
Data for user '3935173',  con 190 observaciones, 2 fraudes
Se agregaron 190 nodos a la red
Se agregaron 3491 ejes a la red
Se guardo el Grapho red_navegador_caract_3935173.gml'
{'       4699985': ('            36', '             1', '           9.0'), '       2400976': ('            10', '             2', '           5.0'), '       4285617': ('            37', '             0', '          30.0'), '       4301928': ('            50', '            26', '          42.0'), '       7669090': ('             6', '             6', '           6.0'), '        150073': ('            56', '            67', '          48.0'), '       3408431': ('            10', '             2', '           2.0'), '       4289668': ('            80', '             1', '          36.0'), '       6726081': ('             6', '             3', '           2.0'), '       4414533': ('     

Se agregaron 164 nodos a la red
Se agregaron 6413 ejes a la red
Se guardo el Grapho red_ip_address_4699985.gml'
User 2400976
Data for user '2400976',  con 60 observaciones, 2 fraudes
Se agregaron 60 nodos a la red
Se agregaron 493 ejes a la red
Se guardo el Grapho red_ip_address_2400976.gml'
User 4285617
Data for user '4285617',  con 126 observaciones, 1 fraudes
Se agregaron 126 nodos a la red
Se agregaron 6673 ejes a la red
Se guardo el Grapho red_ip_address_4285617.gml'
User 4301928
Data for user '4301928',  con 175 observaciones, 2 fraudes
Se agregaron 175 nodos a la red
Se agregaron 13211 ejes a la red
Se guardo el Grapho red_ip_address_4301928.gml'
User 7669090
Data for user '7669090',  con 15 observaciones, 1 fraudes
Se agregaron 15 nodos a la red
Se agregaron 48 ejes a la red
Se guardo el Grapho red_ip_address_7669090.gml'
User 150073
Data for user '150073',  con 195 observaciones, 1 fraudes
Se agregaron 195 nodos a la red
Se agregaron 18145 ejes a la red
Se guardo el Grapho red

Final del archivo. Debe existir un output en la carpeta 'Output_variables'